# MovieLens playground

Local look at `ml-latest-small`. Raw files live under `data/raw_private/movielens/` (gitignored).

Use this notebook to poke around: shapes, joins to `tmdbId`, genres, rating quirks.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("..") / "data" / "raw_private" / "movielens" / "ml-latest-small"

movies = pd.read_csv(DATA / "movies.csv")
ratings = pd.read_csv(DATA / "ratings.csv")
tags = pd.read_csv(DATA / "tags.csv")
links = pd.read_csv(DATA / "links.csv")

for name, df in [
    ("movies", movies),
    ("ratings", ratings),
    ("tags", tags),
    ("links", links),
]:
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} cols → {list(df.columns)}")

movies: 9,742 rows × 3 cols → ['movieId', 'title', 'genres']
ratings: 100,836 rows × 4 cols → ['userId', 'movieId', 'rating', 'timestamp']
tags: 3,683 rows × 4 cols → ['userId', 'movieId', 'tag', 'timestamp']
links: 9,742 rows × 3 cols → ['movieId', 'imdbId', 'tmdbId']


In [2]:
# Nulls + rating basics
display(links.isna().sum())
print("users:", ratings["userId"].nunique())
print("movies with ≥1 rating:", ratings["movieId"].nunique())
print(
    "rating min / max / mean:",
    ratings["rating"].min(),
    ratings["rating"].max(),
    round(ratings["rating"].mean(), 3),
)
ratings["rating"].value_counts().sort_index()

movieId    0
imdbId     0
tmdbId     8
dtype: int64

users: 610
movies with ≥1 rating: 9724
rating min / max / mean: 0.5 5.0 3.502


rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64

In [3]:
# Join movies ↔ TMDb ids (this is the bridge we'll use later)
movies_x = movies.merge(links, on="movieId", how="left")
print("movies missing tmdbId:", movies_x["tmdbId"].isna().sum())
movies_x.head(10)

movies missing tmdbId: 8


,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0
5,6,Heat (1995),Action|Crime|Thriller,113277,949.0
6,7,Sabrina (1995),Comedy|Romance,114319,11860.0
7,8,Tom and Huck (1995),Adventure|Children,112302,45325.0
8,9,Sudden Death (1995),Action,114576,9091.0
9,10,GoldenEye (1995),Action|Adventure|Thriller,113189,710.0


In [4]:
# Genres are pipe-separated — explode a bit
genre_counts = (
    movies["genres"]
    .str.split("|")
    .explode()
    .value_counts()
)
genre_counts.head(15)

genres
Drama          4361
Comedy         3756
Thriller       1894
Action         1828
Romance        1596
Adventure      1263
Crime          1199
Sci-Fi          980
Horror          978
Fantasy         779
Children        664
Animation       611
Mystery         573
Documentary     440
War             382
Name: count, dtype: int64

In [5]:
# Pick a title and see its ratings + tags (playaround)
q = "Toy Story"
hits = movies_x[movies_x["title"].str.contains(q, case=False, na=False)]
display(hits)

mid = hits.iloc[0]["movieId"]
print("ratings for this movie:", (ratings["movieId"] == mid).sum())
display(ratings[ratings["movieId"] == mid]["rating"].describe())
display(tags[tags["movieId"] == mid].head(20))

,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,120363,863.0
7355,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,435761,10193.0


ratings for this movie: 215


count    215.000000
mean       3.920930
std        0.834859
min        0.500000
25%        3.500000
50%        4.000000
75%        4.500000
max        5.000000
Name: rating, dtype: float64

,userId,movieId,tag,timestamp
629,336,1,pixar,1139045764
981,474,1,pixar,1137206825
2886,567,1,fun,1525286013
